### Visualize dimension CCGP on fixed feature 6-tuples over StimOnset and FeedbackOnsetLong
Runs of `scripts/pseudo_decoding/belief_partitions/dim_ccgp_tuple_feat_beliefs.py`, launched by `slurm_launch_dim_ccgp_tuples_full.sh`. Design: `claude_notes/dimension_ccgp_fixed_tuples_proposal.md`. The 8 tuples of `generate_dim_tuples.py` (3 + 3 features, each the rule for >= 2 blocks in >= 7 sessions), on High trials where the preferred feature was chosen and rewarded.

Units are those preference-selective for any of the tuple's features (`pref_99th_dim_tuples_window_filter_drift`, `run_anova2.py` on the tuples' sessions).
- **dimension** folds: train (A_i, A_j) vs (B_i, B_j), test A_k vs B_k
- **random** folds: the second-slot training features swapped, (A_i, B_j) vs (B_i, A_j), same test trials and labels
- **train**: held-out accuracy on the training features
- **CCGP**: accuracy on all trials of the held-out pair

Shuffles are session permuted. Error bands are SE over tuples x folds x splits. Significance bars are the stored p values of `compute_p_vals_for_dim_ccgp_tuples.py`, true vs shuffle per bin and fold type (thin p < 0.05, medium p < 0.01, thick p < 0.001).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import argparse
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import utils.visualization_utils as visualization_utils
from scripts.pseudo_decoding.belief_partitions.belief_partition_configs import *
import scripts.pseudo_decoding.belief_partitions.belief_partitions_io as belief_partitions_io
from scripts.pseudo_decoding.belief_partitions.generate_dim_tuples import TUPLES_PATH
from scripts.pseudo_decoding.belief_partitions.dim_ccgp_tuple_feat_beliefs import MODE, FOLD_TYPES

In [ ]:
BASE_OUTPUT_PATH = "/data/patrick_res/dim_ccgp_tuples"
SIG_UNIT_LEVEL = "pref_99th_dim_tuples_window_filter_drift"
BEH_FILTERS = {"Response": "Correct", "PreferredChosen": True}
EVENTS = ["StimOnset", "FeedbackOnsetLong"]
tuples = pd.read_pickle(TUPLES_PATH)

all_res, all_p_vals = [], []
for event in EVENTS:
    args = argparse.Namespace(
        **BeliefPartitionConfigs()._asdict()
    )
    args.subject = "both"
    args.trial_event = event
    args.mode = MODE
    args.beh_filters = BEH_FILTERS
    args.sig_unit_level = SIG_UNIT_LEVEL
    args.base_output_path = BASE_OUTPUT_PATH

    res = belief_partitions_io.read_dim_ccgp_tuple_results(args, tuples, num_shuffles=10)
    res["event"] = event
    all_res.append(res)
    p_vals = pd.read_pickle(os.path.join(belief_partitions_io.get_dir_name(args, make_dir=False), "dim_ccgp_tuples_p_vals.pickle"))
    p_vals["event"] = event
    all_p_vals.append(p_vals)
res = pd.concat(all_res, ignore_index=True)
p_vals = pd.concat(all_p_vals, ignore_index=True)

# population per tuple, true vs shuffle
res.groupby(["event", "tuple_idx", "dim_a", "dim_b", "shuffle_type"])[["n_sessions", "n_units", "n_dropped_sessions"]].agg(["min", "max"])

In [ ]:
# long format, one row per (run row, measure), with the earlier CCGP figures' colors
MEASURE_TO_DISPLAY = {"train_acc": "train", "ccgp_acc": "CCGP"}
COND_TO_COLOR = {
    "train": "#1b9e77",
    "train shuffle": "#b3e2cd",
    "CCGP": "#d95f02",
    "CCGP shuffle": "#fdcdac",
}
long_res = res.melt(
    id_vars=["event", "tuple_idx", "dim_a", "dim_b", "fold_type", "fold", "split_idx", "shuffle_idx", "shuffle_type", "Time", "TimeIdx"],
    value_vars=list(MEASURE_TO_DISPLAY), var_name="measure", value_name="Accuracy",
)
long_res["condition"] = long_res.measure.map(MEASURE_TO_DISPLAY) + np.where(long_res.shuffle_type == "shuffle", " shuffle", "")

In [ ]:
# event markers, as in visualization_utils.plot_combined_accs
EVENT_LINES = {"StimOnset": [-0.5, 0], "FeedbackOnsetLong": [-0.8, 0]}
EVENT_XLABELS = {"StimOnset": "Time to stimuli appear (s)", "FeedbackOnsetLong": "Time to feedback (s)"}

plt.rcParams.update({'font.size': 16})
width_ratios = [res[res.event == event].Time.nunique() for event in EVENTS]
fig, axs = plt.subplots(len(FOLD_TYPES), len(EVENTS), figsize=(17, 9), sharey=True, width_ratios=width_ratios)
for row, fold_type in enumerate(FOLD_TYPES):
    for col, event in enumerate(EVENTS):
        ax = axs[row, col]
        sub = long_res[(long_res.event == event) & (long_res.fold_type == fold_type)]
        sns.lineplot(
            sub, x="Time", y="Accuracy", hue="condition", hue_order=list(COND_TO_COLOR),
            palette=COND_TO_COLOR, linewidth=3, errorbar="se", ax=ax, legend=(row == 0 and col == len(EVENTS) - 1),
        )
        ax.axhline(0.5, linestyle="dotted", color="black")
        for x in EVENT_LINES[event]:
            ax.axvline(x, color='grey', linestyle='dotted', linewidth=3)
        ax.set_title(f"{fold_type} folds, {event}")
        ax.set_xlabel(EVENT_XLABELS[event])
        ax.set_ylabel("Accuracy")

# significance bars below the lines, one row per measure, in that measure's color
y_min, y_max = axs[0, 0].get_ylim()
interval = (y_max - y_min) / 15
for row, fold_type in enumerate(FOLD_TYPES):
    for col, event in enumerate(EVENTS):
        event_p_vals = p_vals[(p_vals.event == event) & (p_vals.fold_type == fold_type)]
        for i, (measure, display_name) in enumerate(MEASURE_TO_DISPLAY.items()):
            visualization_utils.plot_sig_bars(event_p_vals[event_p_vals.measure == measure], y_min - interval * (i + 0.5), axs[row, col], color=COND_TO_COLOR[display_name])
axs[0, 0].set_ylim(bottom=y_min - interval * (len(MEASURE_TO_DISPLAY) + 0.5))

legend = axs[0, -1].legend(fontsize=12, loc="upper left", bbox_to_anchor=(1.02, 1))
for line in legend.get_lines():
    line.set_linewidth(6)
visualization_utils.format_plot(list(axs.ravel()))
fig.tight_layout()

In [ ]:
# mean accuracy per bin and condition, next to the p values, per fold type
for event in EVENTS:
    for fold_type in FOLD_TYPES:
        sub = long_res[(long_res.event == event) & (long_res.fold_type == fold_type)]
        means = sub.groupby(["Time", "condition"]).Accuracy.mean().unstack()[list(COND_TO_COLOR)]
        ps = p_vals[(p_vals.event == event) & (p_vals.fold_type == fold_type)].pivot(index="Time", columns="measure", values="p")
        ps = ps.rename(columns=lambda m: f"p {MEASURE_TO_DISPLAY[m]}")
        print(event, fold_type)
        display(means.join(ps).round(3))

In [ ]:
# true minus mean shuffle per tuple and fold type, averaged over bins and folds
gaps = res.groupby(["event", "tuple_idx", "dim_a", "dim_b", "fold_type", "shuffle_type"])[["train_acc", "ccgp_acc"]].mean().unstack("shuffle_type")
gaps = pd.DataFrame({
    "train gap": gaps[("train_acc", "true")] - gaps[("train_acc", "shuffle")],
    "CCGP gap": gaps[("ccgp_acc", "true")] - gaps[("ccgp_acc", "shuffle")],
}).unstack("fold_type")
gaps.round(4)

In [ ]:
# per-tuple CCGP over time: dimension vs random folds (true runs), with the dimension folds' shuffle mean
FOLD_TYPE_TO_COLOR = {"dimension": "#d95f02", "random": "#7570b3"}
for event in EVENTS:
    fig, axs = plt.subplots(2, 4, figsize=(20, 8), sharex=True, sharey=True)
    for ax, (tuple_idx, row) in zip(axs.ravel(), tuples.iterrows()):
        sub = res[(res.event == event) & (res.tuple_idx == tuple_idx)]
        sns.lineplot(
            sub[sub.shuffle_type == "true"], x="Time", y="ccgp_acc", hue="fold_type", hue_order=FOLD_TYPES,
            palette=FOLD_TYPE_TO_COLOR, errorbar="se", ax=ax, legend=(tuple_idx == 0),
        )
        shuffle_mean = sub[(sub.shuffle_type == "shuffle") & (sub.fold_type == "dimension")].groupby("Time").ccgp_acc.mean()
        ax.plot(shuffle_mean.index, shuffle_mean.values, color="grey", linestyle="--", label="dimension shuffle")
        ax.axhline(0.5, linestyle="dotted", color="black")
        ax.set_title(f"{tuple_idx}: {', '.join(row.feats_a)} | {', '.join(row.feats_b)}", fontsize=10)
        ax.set_ylabel("CCGP accuracy")
    fig.suptitle(event)
    fig.tight_layout()